# Fase 3 — Preparação dos Dados

**Projeto**: Mineração de Padrões Frequentes em Acidentes de Rodovias Federais  
**Grupo 18**: Erik Roberto Reis Neves, Gabriel Campos Prudente, Felipe Silva Fraga Damasceno  
**Disciplina**: Mineração de Dados — UFMG  

---

## Objetivos desta fase
1. Tratar valores ausentes e inconsistências nos dados
2. Limpar e padronizar variáveis categóricas
3. Criar features derivadas (engenharia de atributos)
4. Construir a representação transacional (one-hot encoding) para FP-Growth
5. Filtrar itens por frequência (1% ≤ freq ≤ 99%)
6. Salvar dados processados para as fases seguintes

**Alinhamento CRISP-DM**: *Data Preparation*

## 3.0 Setup — Importações e Carregamento

Reutiliza a configuração e funções da Fase 1.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURAÇÕES DE VISUALIZAÇÃO
# ============================================================
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Imports realizados com sucesso.')

In [ ]:
# ============================================================
# CONFIGURAÇÃO CENTRAL (replicada da Fase 1)
# ============================================================
DATA_DIR = Path('../data/')
PROCESSED_DIR = Path('../data/processed/')
OUTPUT_DIR = Path('../outputs/figuras/')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'por_ocorrencia': {
        'file': 'datatran2026_agrupados_por_ocorrencia.csv',
        'granularity': 'ocorrência (1 linha = 1 acidente)',
        'id_col': 'id',
    },
    'por_pessoa': {
        'file': 'acidentes2026_agrupados_por_pessoa.csv',
        'granularity': 'pessoa (1 linha = 1 pessoa envolvida)',
        'id_col': 'pesid',
    },
    'todas_causas': {
        'file': 'acidentes2026_todas_causas_tipos.csv',
        'granularity': 'causa x pessoa (explode multicausa)',
        'id_col': 'pesid',
    },
}

# >>> DATASET ATIVO <<<
ACTIVE_DATASET = 'por_ocorrencia'
FILTRO_UF = 'MG'
SEP = ';'
ENCODING = 'latin-1'

# Colunas categóricas comuns para análise
COLUNAS_CATEGORICAS = [
    'dia_semana', 'causa_acidente', 'tipo_acidente',
    'classificacao_acidente', 'fase_dia', 'sentido_via',
    'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo',
]

COLUNAS_NUMERICAS = ['pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'feridos', 'veiculos']
COLUNAS_TEMPORAIS = ['data_inversa', 'horario']

In [ ]:
# ============================================================
# CARREGAMENTO
# ============================================================
info = DATASETS[ACTIVE_DATASET]
path = DATA_DIR / info['file']
df = pd.read_csv(path, sep=SEP, encoding=ENCODING, low_memory=False)

if FILTRO_UF:
    df = df[df['uf'] == FILTRO_UF].copy()

print(f'Dataset: {info["file"]}')
print(f'Granularidade: {info["granularity"]}')
print(f'Filtro UF: {FILTRO_UF}')
print(f'Registros: {len(df):,}')
print(f'Colunas: {df.shape[1]}')
print(f'\nFormato original do DataFrame:')
df.head(3)

---

## 3.1 Tratamento de Valores Ausentes

Estratégia:
- Colunas com > 30% nulos → avaliar exclusão
- `classificacao_acidente` com NA → tratar como 'Não Informado' ou remover
- Categóricas restantes → preencher com 'Não Informado'
- Numéricas → preencher com mediana

In [ ]:
# Diagnóstico de valores nulos
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
nulos_df = pd.DataFrame({'nulos': nulos, 'nulos_%': nulos_pct})
nulos_df = nulos_df[nulos_df['nulos'] > 0].sort_values('nulos_%', ascending=False)

if len(nulos_df) > 0:
    print('=== Colunas com valores nulos ===')
    print(nulos_df)
    print()
    
    # Identificar colunas com > 30% nulos para possível exclusão
    colunas_muitos_nulos = nulos_df[nulos_df['nulos_%'] > 30].index.tolist()
    if colunas_muitos_nulos:
        print(f'⚠️  Colunas com > 30% nulos (candidatas à exclusão): {colunas_muitos_nulos}')
    
    # Tratar classificacao_acidente
    if 'classificacao_acidente' in df.columns:
        n_nulos_classif = df['classificacao_acidente'].isnull().sum()
        if n_nulos_classif > 0:
            print(f'\n classificacao_acidente: {n_nulos_classif} nulos → removendo esses registros')
            df = df.dropna(subset=['classificacao_acidente']).copy()
    
    # Preencher categóricas com 'Não Informado'
    for col in COLUNAS_CATEGORICAS:
        if col in df.columns and df[col].isnull().any():
            n_before = df[col].isnull().sum()
            df[col] = df[col].fillna('Não Informado')
            print(f'  {col}: {n_before} nulos → preenchidos com "Não Informado"')
    
    # Preencher numéricas com mediana
    for col in COLUNAS_NUMERICAS:
        if col in df.columns and df[col].isnull().any():
            mediana = df[col].median()
            n_before = df[col].isnull().sum()
            df[col] = df[col].fillna(mediana)
            print(f'  {col}: {n_before} nulos → preenchidos com mediana ({mediana})')
else:
    print('✅ Nenhuma coluna possui valores nulos.')

# Verificação final
nulos_restantes = df[COLUNAS_CATEGORICAS + COLUNAS_NUMERICAS].isnull().sum().sum()
print(f'\n✅ Nulos restantes nas colunas de interesse: {nulos_restantes}')
print(f'   Registros após tratamento: {len(df):,}')

---

## 3.2 Limpeza e Padronização

- Remover espaços extras em strings
- Tratar `km` (vírgula como decimal → float)
- Tratar `latitude`/`longitude` (vírgula → ponto)
- Tratar `tracado_via` (valores compostos com `;`)

In [ ]:
# 3.2.1 — Remover espaços extras em todas as colunas string
print('=== Limpeza de espaços extras ===')
str_cols = df.select_dtypes(include=['object', 'string']).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()
print(f'  {len(str_cols)} colunas de texto limpas')

# 3.2.2 — Converter km (vírgula → ponto → float)
if 'km' in df.columns:
    # Forçar conversão independente do dtype (após strip, sempre object)
    df['km'] = df['km'].astype(str).str.replace(',', '.', regex=False)
    df['km'] = pd.to_numeric(df['km'], errors='coerce')
    print(f'  km: convertido para float (nulos após conversão: {df["km"].isnull().sum()})')
    # Preencher nulos de km com mediana
    if df['km'].isnull().any():
        df['km'] = df['km'].fillna(df['km'].median())

# 3.2.3 — Converter latitude/longitude
for coord_col in ['latitude', 'longitude']:
    if coord_col in df.columns:
        df[coord_col] = df[coord_col].astype(str).str.replace(',', '.', regex=False)
        df[coord_col] = pd.to_numeric(df[coord_col], errors='coerce')
        print(f'  {coord_col}: convertido para float')

print('\n✅ Limpeza e padronização concluída')

In [ ]:
# 3.2.4 — Tratamento de tracado_via (valores compostos com ';')
# Conforme decisão do plano: fazer SPLIT e usar PRIMEIRO componente como valor principal
# Ex: "Aclive;Curva" → mantemos como string composta para one-hot
# Na fase transacional, cada valor único de tracado_via vira um item

print('=== Análise de tracado_via ===')
if 'tracado_via' in df.columns:
    # Contar valores únicos antes do tratamento
    n_unicos_antes = df['tracado_via'].nunique()
    print(f'  Valores únicos antes: {n_unicos_antes}')
    
    # Verificar quais contêm ';'
    compostos = df['tracado_via'].str.contains(';', na=False)
    n_compostos = compostos.sum()
    print(f'  Registros com valores compostos (;): {n_compostos} ({n_compostos/len(df)*100:.1f}%)')
    
    # Mostrar top valores compostos
    if n_compostos > 0:
        print(f'\n  Top 10 valores de tracado_via:')
        print(df['tracado_via'].value_counts().head(10).to_string())
    
    # Decisão: manter como string composta para mineração
    # Cada combinação única será um item no one-hot encoding
    print(f'\n  ✅ Mantido como string composta (cada combinação = 1 item no FP-Growth)')

In [ ]:
# 3.2.5 — Verificação de categorias e padronização
print('=== Valores únicos por coluna categórica ===')
for col in COLUNAS_CATEGORICAS:
    if col in df.columns:
        n_unicos = df[col].nunique()
        print(f'\n  {col} ({n_unicos} valores únicos):')
        vc = df[col].value_counts()
        # Mostrar top 5 + categorias raras
        for val, cnt in vc.head(5).items():
            pct = cnt / len(df) * 100
            print(f'    {val}: {cnt} ({pct:.1f}%)')
        if len(vc) > 5:
            print(f'    ... +{len(vc)-5} outros')

---

## 3.3 Feature Engineering

Criar novas variáveis derivadas para enriquecer a mineração de padrões.

In [ ]:
# 3.3.1 — Converter data e extrair mês
if 'data_inversa' in df.columns:
    df['data_inversa'] = pd.to_datetime(df['data_inversa'], errors='coerce')
    df['mes'] = df['data_inversa'].dt.month
    df['mes'] = df['mes'].apply(lambda x: f'Mes_{int(x):02d}' if pd.notna(x) else 'Não Informado')
    print(f'✅ mes criado: {df["mes"].nunique()} valores únicos')
    print(df['mes'].value_counts().sort_index())

print()

In [ ]:
# 3.3.2 — Faixa horária
def classificar_faixa_horaria(horario_str):
    """Discretiza o horário em faixas: Madrugada, Manhã, Tarde, Noite."""
    try:
        hora = int(str(horario_str).split(':')[0])
        if 0 <= hora < 6:
            return 'Madrugada'
        elif 6 <= hora < 12:
            return 'Manhã'
        elif 12 <= hora < 18:
            return 'Tarde'
        else:
            return 'Noite'
    except:
        return 'Não Informado'

if 'horario' in df.columns:
    df['faixa_horaria'] = df['horario'].apply(classificar_faixa_horaria)
    print(f'✅ faixa_horaria criada:')
    print(df['faixa_horaria'].value_counts())

print()

In [ ]:
# 3.3.3 — Fim de semana (booleano)
# Aceitar tanto 'sábado' quanto 'sabado' (com/sem acento)

if 'dia_semana' in df.columns:
    dia_lower = df['dia_semana'].str.lower().str.strip()
    df['fim_de_semana'] = dia_lower.apply(
        lambda x: 'Sim' if x in ['sábado', 'sabado', 'domingo'] or 'bado' in x or 'mingo' in x else 'Não'
    )
    print(f'✅ fim_de_semana criada:')
    print(df['fim_de_semana'].value_counts())

print()

In [ ]:
# 3.3.4 — Gravidade binária (1 se fatal, 0 caso contrário)
if 'classificacao_acidente' in df.columns:
    df['gravidade_binaria'] = (
        df['classificacao_acidente'] == 'Com Vítimas Fatais'
    ).map({True: 'Fatal', False: 'Não Fatal'})
    print(f'✅ gravidade_binaria criada:')
    print(df['gravidade_binaria'].value_counts())

print()

In [ ]:
# 3.3.5 — Faixa de KM (discretizar com base nos percentis)
if 'km' in df.columns:
    # Garantir que km é numérico
    df['km'] = pd.to_numeric(df['km'], errors='coerce')
    km_valid = df['km'].dropna()
    if len(km_valid) > 0:
        p25 = km_valid.quantile(0.25)
        p50 = km_valid.quantile(0.50)
        p75 = km_valid.quantile(0.75)
    print(f'Percentis de km: P25={p25:.1f}, P50={p50:.1f}, P75={p75:.1f}')
    
    # Usar bins baseados nos dados
    bins = [0, 50, 100, 200, 400, float('inf')]
    labels = ['KM_0-50', 'KM_50-100', 'KM_100-200', 'KM_200-400', 'KM_400+']
    
    df['faixa_km'] = pd.cut(df['km'], bins=bins, labels=labels, right=False)
    df['faixa_km'] = df['faixa_km'].astype(str)
    df['faixa_km'] = df['faixa_km'].replace('nan', 'Não Informado')
    
    print(f'\n✅ faixa_km criada:')
    print(df['faixa_km'].value_counts().sort_index())

print()

In [ ]:
# 3.3.6 — Resumo das features engenheiradas
features_novas = ['mes', 'faixa_horaria', 'fim_de_semana', 'gravidade_binaria', 'faixa_km']
features_criadas = [f for f in features_novas if f in df.columns]

print(f'=== Resumo de Features Engenheiradas ===')
print(f'Features criadas: {len(features_criadas)}')
for f in features_criadas:
    print(f'  {f}: {df[f].nunique()} valores únicos')

print(f'\nDataFrame atual: {len(df):,} registros × {df.shape[1]} colunas')

---

## 3.4 Seleção de Colunas para Mineração

Definir quais colunas serão usadas na representação transacional para o FP-Growth.

In [ ]:
# ============================================================
# SELEÇÃO FLEXÍVEL DE COLUNAS TRANSACIONAIS
# ============================================================
# >>> EDITE AQUI para adicionar/remover colunas da mineração <<<

COLUNAS_TRANSACIONAIS = [
    # --- Colunas originais ---
    'dia_semana',
    'fase_dia',
    'condicao_metereologica',
    'tipo_pista',
    'tracado_via',
    'uso_solo',
    'causa_acidente',
    'tipo_acidente',
    'classificacao_acidente',
    # --- Features engenheiradas ---
    'faixa_horaria',
    'fim_de_semana',
    'gravidade_binaria',
    # --- Opcionais (descomentar conforme necessidade) ---
    # 'faixa_km',           # pode gerar muitos itens
    # 'mes',                # análise temporal
]

# Filtrar apenas colunas que existem no DataFrame
colunas_disponiveis = [c for c in COLUNAS_TRANSACIONAIS if c in df.columns]
colunas_ausentes = [c for c in COLUNAS_TRANSACIONAIS if c not in df.columns]

print(f'=== Colunas Selecionadas para Mineração ===')
print(f'Colunas disponíveis: {len(colunas_disponiveis)}')
for c in colunas_disponiveis:
    n_unicos = df[c].nunique()
    print(f'  ✅ {c}: {n_unicos} valores únicos')

if colunas_ausentes:
    print(f'\n⚠️  Colunas não encontradas: {colunas_ausentes}')

# Verificar nulos nas colunas selecionadas
nulos_transac = df[colunas_disponiveis].isnull().sum()
nulos_transac = nulos_transac[nulos_transac > 0]
if len(nulos_transac) > 0:
    print(f'\n⚠️  Nulos encontrados:')
    print(nulos_transac)
else:
    print(f'\n✅ Zero nulos nas colunas transacionais')

---

## 3.5 Construção da Representação Transacional (One-Hot Encoding)

Transformar cada registro em uma "transação" binária para o FP-Growth.  
Cada coluna vira itens no formato `nome_coluna=valor`.

In [ ]:
def criar_representacao_transacional(df, colunas):
    """
    Cria DataFrame binário (one-hot) para FP-Growth.
    Cada coluna categórica vira itens no formato 'nome_coluna=valor'.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame com os dados.
    colunas : list
        Lista de colunas categóricas a transformar.
    
    Returns
    -------
    pd.DataFrame
        DataFrame binário (boolean) com itens como colunas.
    """
    df_trans = pd.DataFrame(index=df.index)
    
    for col in colunas:
        if col in df.columns:
            dummies = pd.get_dummies(df[col], prefix=col, dtype=bool)
            df_trans = pd.concat([df_trans, dummies], axis=1)
    
    return df_trans


# Criar representação transacional
df_onehot = criar_representacao_transacional(df, colunas_disponiveis)

print(f'=== Representação Transacional ===')
print(f'Registros (transações): {df_onehot.shape[0]:,}')
print(f'Itens (colunas binárias): {df_onehot.shape[1]:,}')
print(f'Tipo de dados: {df_onehot.dtypes.unique()}')
print(f'Memória: {df_onehot.memory_usage(deep=True).sum() / 1e6:.1f} MB')

# Amostra
print(f'\nAmostra (5 primeiras linhas, 10 primeiras colunas):')
df_onehot.iloc[:5, :10]

---

## 3.6 Controle de Granularidade de Itens

Filtrar itens por frequência:
- Remover itens com frequência < 1% (muito raros)
- Remover itens com frequência > 99% (triviais)

In [ ]:
# Calcular frequência de cada item
freq = df_onehot.mean()

# Limites de filtragem
MIN_FREQ = 0.01  # 1%
MAX_FREQ = 0.99  # 99%

# Itens muito raros (< 1%)
itens_raros = freq[freq < MIN_FREQ].sort_values()
print(f'=== Itens muito raros (freq < {MIN_FREQ*100:.0f}%) ===')
print(f'Total: {len(itens_raros)}')
if len(itens_raros) > 0:
    for item, f in itens_raros.items():
        print(f'  {item}: {f*100:.2f}%')

print()

# Itens muito frequentes (> 99%)
itens_triviais = freq[freq > MAX_FREQ].sort_values(ascending=False)
print(f'=== Itens triviais (freq > {MAX_FREQ*100:.0f}%) ===')
print(f'Total: {len(itens_triviais)}')
if len(itens_triviais) > 0:
    for item, f in itens_triviais.items():
        print(f'  {item}: {f*100:.2f}%')

In [ ]:
# Aplicar filtragem
itens_validos = freq[(freq >= MIN_FREQ) & (freq <= MAX_FREQ)].index
df_onehot_filtrado = df_onehot[itens_validos]

print(f'=== Resultado da Filtragem ===')
print(f'Itens antes: {df_onehot.shape[1]}')
print(f'Itens removidos (raros): {len(itens_raros)}')
print(f'Itens removidos (triviais): {len(itens_triviais)}')
print(f'Itens depois: {df_onehot_filtrado.shape[1]}')
print(f'Transações: {df_onehot_filtrado.shape[0]:,}')
print(f'Memória: {df_onehot_filtrado.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Visualização da distribuição de frequências dos itens
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma de frequências
freq_filtrado = df_onehot_filtrado.mean().sort_values(ascending=False)
axes[0].hist(freq_filtrado, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(x=MIN_FREQ, color='red', linestyle='--', label=f'Min ({MIN_FREQ*100:.0f}%)')
axes[0].axvline(x=MAX_FREQ, color='orange', linestyle='--', label=f'Max ({MAX_FREQ*100:.0f}%)')
axes[0].set_xlabel('Frequência (proporção)')
axes[0].set_ylabel('Número de itens')
axes[0].set_title('Distribuição de Frequências dos Itens')
axes[0].legend()

# Top 20 itens mais frequentes
top20 = freq_filtrado.head(20)
colors = ['#e74c3c' if 'Fatal' in item or 'Fatais' in item 
          else '#2ecc71' if 'Sem' in item 
          else '#3498db' for item in top20.index]
axes[1].barh(range(len(top20)), top20.values, color=colors, alpha=0.8)
axes[1].set_yticks(range(len(top20)))
axes[1].set_yticklabels([s.replace('_', '\n', 1) if len(s) > 30 else s for s in top20.index], fontsize=9)
axes[1].set_xlabel('Frequência (proporção)')
axes[1].set_title('Top 20 Itens Mais Frequentes')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'distribuicao_itens_transacionais.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'\n✅ Visualização salva em {OUTPUT_DIR / "distribuicao_itens_transacionais.png"}')

---

## 3.7 Salvamento dos Dados Processados

Salvar os dados limpos e a representação transacional para uso nas fases seguintes.

In [ ]:
# Salvar DataFrame limpo
path_df_limpo = PROCESSED_DIR / 'df_limpo.pkl'
df.to_pickle(path_df_limpo)
print(f'✅ DataFrame limpo salvo: {path_df_limpo}')
print(f'   {len(df):,} registros × {df.shape[1]} colunas')

# Salvar representação transacional filtrada
path_transacional = PROCESSED_DIR / 'transacional.pkl'
df_onehot_filtrado.to_pickle(path_transacional)
print(f'\n✅ Representação transacional salva: {path_transacional}')
print(f'   {df_onehot_filtrado.shape[0]:,} transações × {df_onehot_filtrado.shape[1]} itens')

# Salvar também a versão não filtrada (para referência)
path_transacional_full = PROCESSED_DIR / 'transacional_completo.pkl'
df_onehot.to_pickle(path_transacional_full)
print(f'\n✅ Representação transacional completa (sem filtro): {path_transacional_full}')
print(f'   {df_onehot.shape[0]:,} transações × {df_onehot.shape[1]} itens')

# Salvar metadados da preparação
metadata = {
    'dataset_ativo': ACTIVE_DATASET,
    'filtro_uf': FILTRO_UF,
    'n_registros': len(df),
    'n_colunas_df': df.shape[1],
    'colunas_transacionais': colunas_disponiveis,
    'n_itens_total': df_onehot.shape[1],
    'n_itens_filtrado': df_onehot_filtrado.shape[1],
    'n_itens_raros_removidos': len(itens_raros),
    'n_itens_triviais_removidos': len(itens_triviais),
    'min_freq': MIN_FREQ,
    'max_freq': MAX_FREQ,
    'features_engenheiradas': features_criadas,
}

import json
path_metadata = PROCESSED_DIR / 'preparacao_metadata.json'
with open(path_metadata, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f'\n✅ Metadados salvos: {path_metadata}')

---

## 3.8 Verificação Final e Resumo

In [ ]:
# Resumo final da preparação
print('=' * 60)
print('RESUMO DA FASE 3 — PREPARAÇÃO DOS DADOS')
print('=' * 60)

print(f'\nDataset: {ACTIVE_DATASET} ({FILTRO_UF})')
print(f'Registros: {len(df):,}')
print(f'Colunas totais: {df.shape[1]}')

# Verificar nulos nas colunas usadas
nulos_final = df[colunas_disponiveis].isnull().sum().sum()
print(f'\n1. Valores nulos nas colunas transacionais: {nulos_final}')
status_nulos = '✅' if nulos_final == 0 else '❌'
print(f'   {status_nulos} Zero valores nulos nas colunas usadas')

# Features categóricas padronizadas
print(f'\n2. Features categóricas padronizadas:')
for col in colunas_disponiveis:
    print(f'   ✅ {col}: {df[col].nunique()} categorias')

# Features derivadas
print(f'\n3. Features derivadas criadas e validadas:')
for f in features_criadas:
    print(f'   ✅ {f}: {df[f].nunique()} valores')

# Representação transacional
print(f'\n4. Representação transacional one-hot:')
print(f'   ✅ {df_onehot.shape[0]:,} transações × {df_onehot.shape[1]} itens')

# Filtragem de itens
print(f'\n5. Filtragem de itens ({MIN_FREQ*100:.0f}% ≤ freq ≤ {MAX_FREQ*100:.0f}%):')
print(f'   ✅ {df_onehot.shape[1]} → {df_onehot_filtrado.shape[1]} itens')
print(f'   Removidos: {len(itens_raros)} raros + {len(itens_triviais)} triviais')

# Dados salvos
import os
print(f'\n6. Dados processados salvos em data/processed/:')
for f in PROCESSED_DIR.iterdir():
    size = os.path.getsize(f) / 1e6
    print(f'   ✅ {f.name} ({size:.1f} MB)')

print(f'\n{"=" * 60}')
print('✅ FASE 3 CONCLUÍDA COM SUCESSO')
print(f'{"=" * 60}')